In [36]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# Base de données, id_user et moyenne des notes par catégories

In [37]:
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

In [38]:
movies.head()

,movie_id,title,rates,nb_rates,mean,year,genre_list,Action,Adventure,Animation,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,0.0,0.0,NaN,1995,"['Adventure', 'Animation', 'Children', 'Comedy...",0,1,1,...,1,0,0,0,0,0,0,0,0,0
1,2,Jumanji,103912.0,26449.0,3.928769,1995,"['Adventure', 'Children', 'Fantasy']",0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men,38599.5,12032.0,3.208070,1995,"['Comedy', 'Romance']",0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,24541.5,7790.0,3.150385,1995,"['Comedy', 'Drama', 'Romance']",0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II,5046.0,1764.0,2.860544,1995,['Comedy'],0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:
data = ratings.merge(movies, on="movie_id")
data.head()

,user_id,movie_id,rating,title,rates,nb_rates,mean,year,genre_list,Action,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,122,5.0,Boomerang,238.0,84.0,2.833333,1992,"['Comedy', 'Romance']",0,...,0,0,0,0,0,1,0,0,0,0
1,1,185,5.0,"Net, The",1346.5,402.0,3.349502,1995,"['Action', 'Crime', 'Thriller']",1,...,0,0,0,0,0,0,0,1,0,0
2,1,231,5.0,Dumb & Dumber,798.0,290.0,2.751724,1994,['Comedy'],0,...,0,0,0,0,0,0,0,0,0,0
3,1,292,5.0,Outbreak,6682.5,2002.0,3.337912,1995,"['Action', 'Drama', 'Sci-Fi', 'Thriller']",1,...,0,0,0,0,0,0,1,1,0,0
4,1,316,5.0,Stargate,1881.5,629.0,2.991256,1994,"['Action', 'Adventure', 'Sci-Fi']",1,...,0,0,0,0,0,0,1,0,0,0


In [40]:
genre_cols = movies.columns.difference(
    ["movie_id", "title", "rates", "nb_rates", "mean", "year", "genre_list"]
)

In [41]:
for genre in genre_cols:
    data[f"{genre}_weighted"] = data["rating"] * data[genre]

In [42]:
data.head()

,user_id,movie_id,rating,title,rates,nb_rates,mean,year,genre_list,Action,...,Fantasy_weighted,Film-Noir_weighted,Horror_weighted,Musical_weighted,Mystery_weighted,Romance_weighted,Sci-Fi_weighted,Thriller_weighted,War_weighted,Western_weighted
0,1,122,5.0,Boomerang,238.0,84.0,2.833333,1992,"['Comedy', 'Romance']",0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0
1,1,185,5.0,"Net, The",1346.5,402.0,3.349502,1995,"['Action', 'Crime', 'Thriller']",1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
2,1,231,5.0,Dumb & Dumber,798.0,290.0,2.751724,1994,['Comedy'],0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,292,5.0,Outbreak,6682.5,2002.0,3.337912,1995,"['Action', 'Drama', 'Sci-Fi', 'Thriller']",1,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,0.0,0.0
4,1,316,5.0,Stargate,1881.5,629.0,2.991256,1994,"['Action', 'Adventure', 'Sci-Fi']",1,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0


In [43]:
user_profiles = data.groupby("user_id").agg({
    **{f"{genre}_weighted": "sum" for genre in genre_cols},
    **{genre: "sum" for genre in genre_cols}
}).reset_index()
user_profiles.head()

,user_id,Action_weighted,Adventure_weighted,Animation_weighted,Children's_weighted,Comedy_weighted,Crime_weighted,Documentary_weighted,Drama_weighted,Fantasy_weighted,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,50.0,30.0,20.0,0.0,55.0,10.0,0.0,30.0,10.0,...,2,0,0,3,0,5,5,5,2,0
1,2,46.0,34.0,0.0,0.0,12.0,2.0,0.0,23.0,3.0,...,1,0,1,1,1,4,6,7,3,1
2,3,27.0,22.0,8.0,0.0,40.0,12.0,0.0,83.5,12.5,...,3,1,0,1,4,12,0,6,6,3
3,4,67.0,55.0,15.0,0.0,70.0,21.0,0.0,64.0,23.0,...,5,0,1,3,0,8,8,11,3,3
4,5,9.0,24.0,1.0,0.0,104.0,47.0,0.0,256.0,15.0,...,4,1,6,4,4,26,8,14,6,0


In [44]:
for genre in genre_cols:
    user_profiles[f"mean_{genre}"] = (
        user_profiles[f"{genre}_weighted"] /
        user_profiles[genre].replace(0, np.nan)
    )

In [45]:
user_profiles.fillna(0, inplace=True)
user_profiles.head()

,user_id,Action_weighted,Adventure_weighted,Animation_weighted,Children's_weighted,Comedy_weighted,Crime_weighted,Documentary_weighted,Drama_weighted,Fantasy_weighted,...,mean_Fantasy,mean_Film-Noir,mean_Horror,mean_Musical,mean_Mystery,mean_Romance,mean_Sci-Fi,mean_Thriller,mean_War,mean_Western
0,1,50.0,30.0,20.0,0.0,55.0,10.0,0.0,30.0,10.0,...,5.000000,0.0,0.000000,5.00,0.00,5.000000,5.000,5.000000,5.000000,0.000000
1,2,46.0,34.0,0.0,0.0,12.0,2.0,0.0,23.0,3.0,...,3.000000,0.0,3.000000,3.00,2.00,2.750000,3.500,2.857143,3.666667,5.000000
2,3,27.0,22.0,8.0,0.0,40.0,12.0,0.0,83.5,12.5,...,4.166667,4.0,0.000000,3.00,4.50,4.208333,0.000,4.083333,4.083333,3.666667
3,4,67.0,55.0,15.0,0.0,70.0,21.0,0.0,64.0,23.0,...,4.600000,0.0,3.000000,5.00,0.00,3.750000,4.250,3.909091,5.000000,4.333333
4,5,9.0,24.0,1.0,0.0,104.0,47.0,0.0,256.0,15.0,...,3.750000,5.0,4.333333,3.75,4.25,3.769231,3.375,4.000000,4.000000,0.000000


In [46]:
mean_cols = [f"mean_{genre}" for genre in genre_cols]
user_profiles = user_profiles[["user_id"] + mean_cols]
user_profiles.head()

,user_id,mean_Action,mean_Adventure,mean_Animation,mean_Children's,mean_Comedy,mean_Crime,mean_Documentary,mean_Drama,mean_Fantasy,mean_Film-Noir,mean_Horror,mean_Musical,mean_Mystery,mean_Romance,mean_Sci-Fi,mean_Thriller,mean_War,mean_Western
0,1,5.000000,5.000000,5.0,0.0,5.000000,5.000000,0.0,5.000000,5.000000,0.0,0.000000,5.00,0.00,5.000000,5.000,5.000000,5.000000,0.000000
1,2,3.285714,3.400000,0.0,0.0,3.000000,2.000000,0.0,3.285714,3.000000,0.0,3.000000,3.00,2.00,2.750000,3.500,2.857143,3.666667,5.000000
2,3,3.857143,3.666667,4.0,0.0,3.636364,4.000000,0.0,4.175000,4.166667,4.0,0.000000,3.00,4.50,4.208333,0.000,4.083333,4.083333,3.666667
3,4,3.941176,4.230769,5.0,0.0,3.684211,4.200000,0.0,4.571429,4.600000,0.0,3.000000,5.00,0.00,3.750000,4.250,3.909091,5.000000,4.333333
4,5,1.800000,3.000000,1.0,0.0,3.586207,4.272727,0.0,4.000000,3.750000,5.0,4.333333,3.75,4.25,3.769231,3.375,4.000000,4.000000,0.000000


# Id_type

In [48]:
from sklearn.metrics import silhouette_score

for k in range(2, 16):
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(user_profiles[mean_cols])
    score = silhouette_score(user_profiles[mean_cols], labels)
    print(f"k={k} -> silhouette={score:.4f}")

k=2 -> silhouette=0.2137
k=3 -> silhouette=0.2021
k=4 -> silhouette=0.2003
k=5 -> silhouette=0.2115
k=6 -> silhouette=0.2129
k=7 -> silhouette=0.2031
k=8 -> silhouette=0.2244
k=9 -> silhouette=0.2210
k=10 -> silhouette=0.2254
k=11 -> silhouette=0.2283
k=12 -> silhouette=0.1955
k=13 -> silhouette=0.1979
k=14 -> silhouette=0.1990
k=15 -> silhouette=0.1998


In [49]:
kmeans = KMeans(n_clusters=11, random_state=42)
user_profiles["id_type"] = kmeans.fit_predict(user_profiles[mean_cols])
user_profiles.head()

,user_id,mean_Action,mean_Adventure,mean_Animation,mean_Children's,mean_Comedy,mean_Crime,mean_Documentary,mean_Drama,mean_Fantasy,mean_Film-Noir,mean_Horror,mean_Musical,mean_Mystery,mean_Romance,mean_Sci-Fi,mean_Thriller,mean_War,mean_Western,id_type
0,1,5.000000,5.000000,5.0,0.0,5.000000,5.000000,0.0,5.000000,5.000000,0.0,0.000000,5.00,0.00,5.000000,5.000,5.000000,5.000000,0.000000,5
1,2,3.285714,3.400000,0.0,0.0,3.000000,2.000000,0.0,3.285714,3.000000,0.0,3.000000,3.00,2.00,2.750000,3.500,2.857143,3.666667,5.000000,0
2,3,3.857143,3.666667,4.0,0.0,3.636364,4.000000,0.0,4.175000,4.166667,4.0,0.000000,3.00,4.50,4.208333,0.000,4.083333,4.083333,3.666667,3
3,4,3.941176,4.230769,5.0,0.0,3.684211,4.200000,0.0,4.571429,4.600000,0.0,3.000000,5.00,0.00,3.750000,4.250,3.909091,5.000000,4.333333,4
4,5,1.800000,3.000000,1.0,0.0,3.586207,4.272727,0.0,4.000000,3.750000,5.0,4.333333,3.75,4.25,3.769231,3.375,4.000000,4.000000,0.000000,9


In [50]:
user_profiles.to_csv("user_profiles.csv", index=False)